# Assignment 2 - RAG
Due date: 28.4.26
Students: Yahel Cohen, Gabriel Abramovich

Intro
In this assignment you’ll familiarize yourself with RAG - Retrieval Augmented Generation,
first in theory (just a bit) and then we’ll dive into practicalities. You’ll start querying without
RAG and after adding the RAG component you’ll compare the outcomes. The last steps will
focus on RAG evaluation metrics and improvement cycles.
By the end of this assignment you'll have built a working RAG pipeline, evaluated it across
three different dimensions (not just final-answer correctness), and run improvement cycles
on it. The goal isn't to hit a particular accuracy number - financebench is hard on purpose -
but to develop intuition for which component to fix when results disappoint, which is the skill
that matters in production.

General Instructions
1. Submit one notebook that contains the code and textual answers for all tasks.
2. Some tasks also require xlsx files - pay attention.
3. Zip it all together and include first and last names of both students in the file name.
4. Note: I usually don’t run the notebook, but there should be placeholders for the data
loading, api_key etc. in case I do need to run it (it happens. Rarely, but it happens).
5. Having said that, you need to run it of course, and outputs should be present.

Dataset
We’ll use the financebench dataset. Read the description and read carefully through the
columns and make sure you understand them (you can ignore dataset_subset_label). The
paper can also help you with that.
Notes:
1. The dataset has 3 types of questions: metrics-generated, domain-relevant,
novel-generated. Drop the metrics-generated questions.
2. Some of the urls in the doc_link column are dead. Replace them with links to this
repo. Note that the folder contains more documents than are referenced by the
dataset.

## Task 1 - naive generation
Use the Llama-3.3-70B-Instruct model (via Nebius Token Factory) to answer the first 5
questions (simply sort by financebench_id) of each question_type - 5 domain-relevant, 5
novel-generated.
Note: No retrieval, just the question straight into the model.
For each question, compare the model's output with the expected answer (aka "ground truth"
or "gold answer").
Note: the model might not provide an answer and instead ask for more information, refuse,
or hedge. That's itself a result worth recording.
Look at the answers and identify:
1. Cases where the model refused or asked for more information - why?
2. Cases where the model answered confidently - spot-check against the ground
truth. Is the answer correct? Partially correct? Totally wrong (hallucinated)?
3. Are there patterns by question_type? Do some types fail more than others?
Deliverables:
1. The code.
2. A table (xlsx) with columns: financebench_id | question_type | question |
naive_answer | ground_truth | verdict, where verdict is one of {correct, partially
correct, wrong, refused} based on your manual judgment.
Name the file assignment2_naive_generation.
3. A short written discussion (markdown cell in the notebook) addressing the three
questions above.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import time     
import pandas as pd

load_dotenv()

True

In [2]:
# ---- Task 1: Naive generation (no retrieval) ----
from datasets import load_dataset

# 1. Load FinanceBench, drop metrics-generated, take first 5 of each remaining type
ds = load_dataset("PatronusAI/financebench", split="train").to_pandas()
ds = ds[ds["question_type"] != "metrics-generated"].copy()
sample = (
    ds.sort_values("financebench_id")
      .groupby("question_type", group_keys=False)
      .head(5)
      .reset_index(drop=True)
)
print(f"Sampled {len(sample)} questions: {sample['question_type'].value_counts().to_dict()}")

# 2. Nebius Token Factory client (OpenAI-compatible)
client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.getenv("NEBIUS_API_KEY"),
)
MODEL = "meta-llama/Llama-3.3-70B-Instruct"

SYSTEM_PROMPT = (
    "You are a financial analyst assistant. Answer the user's question as accurately "
    "as you can. If you do not have enough information to answer, say so clearly "
    "instead of guessing."
)

def ask_naive(question: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        temperature=0.0,
        max_tokens=512,
    )
    return resp.choices[0].message.content.strip()

# 3. Run inference
answers = []
for i, row in sample.iterrows():
    print(f"[{i+1}/{len(sample)}] {row['financebench_id']} ({row['question_type']})")
    try:
        answers.append(ask_naive(row["question"]))
    except Exception as e:
        answers.append(f"ERROR: {e}")
    time.sleep(0.4)  # gentle rate-limit

sample["naive_answer"] = answers

# 4. Build deliverable table — verdict left blank for manual judgment
out = (
    sample[["financebench_id", "question_type", "question", "naive_answer", "answer"]]
      .rename(columns={"answer": "ground_truth"})
)
out["verdict"] = ""  # fill manually with: correct | partially correct | wrong | refused

out.to_excel("assignment2_naive_generation.xlsx", index=False)
out

README.md: 0.00B [00:00, ?B/s]

financebench_merged.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/150 [00:00<?, ? examples/s]

Sampled 10 questions: {'domain-relevant': 5, 'novel-generated': 5}
[1/10] financebench_id_00005 (domain-relevant)
[2/10] financebench_id_00070 (domain-relevant)
[3/10] financebench_id_00080 (domain-relevant)
[4/10] financebench_id_00206 (domain-relevant)
[5/10] financebench_id_00215 (domain-relevant)
[6/10] financebench_id_00283 (novel-generated)
[7/10] financebench_id_00288 (novel-generated)
[8/10] financebench_id_00299 (novel-generated)
[9/10] financebench_id_00302 (novel-generated)
[10/10] financebench_id_00382 (novel-generated)


,financebench_id,question_type,question,naive_answer,ground_truth,verdict
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,To determine if Corning has positive working c...,Yes. Corning had a positive working capital am...,
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,To determine whether American Water Works has ...,"No, American Water Works had negative working ...",
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,To determine if PayPal has positive working ca...,Yes. Paypal has a positive working capital of ...,
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"To answer this question, I'll need to look at ...","Since JPM is a financial institution, gross ma...",
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,To determine if Verizon is a capital-intensive...,Yes. Verizon's capital intensity ratio was app...,
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,Pfizer announced plans to spin off its Upjohn ...,77.78,
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,I don't have enough information to answer your...,"Yes, there was a decline of ~42% between FY202...",
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,According to JPMorgan Chase's Q1 2021 earnings...,Corporate. Its net revenue was -$473 million.,
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,To answer whether Pfizer grew its Pre-tax Prof...,"Yes, change in PPNE was positive year over year",
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,I don't have access to real-time data or MGM's...,Las Vegas resorts contributed ~90% of company ...,


Questions:
1. Cases where the model refused or asked for more information - why?
2. Cases where the model answered confidently - spot-check against the ground truth. Is the answer correct? Partially correct? Totally wrong (hallucinated)?
3. Are there patterns by question_type? Do some types fail more than others?

Answers:
1. The cases where the model refused or asked for more information could correlate with information the model could not have memorized like recent quarters, segment level breakdowns.
2. Looking at the generations, there is not a single correct answer which indicates the model abillity to learn things up to a certain granularity. However, the model do had 4 partially-correct answers where he did got the general verdict right (yes/no, capital-intensive/not, etc.) but hallucinates the supporting data used to justify it. He also had 2 cases where he totally fabricated the answer.
3. Yes, there are some patterns that the question_type yields. The domain-relevant are more high-level, so the model hallucinate more than the novel-generated questions where the questions are about narrower slices.

## Task 2 - RAG reminder (5 points)  
Below is a sketch of a simple RAG pipeline. The boxes group into three components - 
indexing (documents → chunk + embed → vector store), retrieval (user query → retrieval), 
and generation. For each component, briefly explain:  
1. How does it contribute to the pipeline?  
2. Where can it fail? Try to think of concrete examples.  
3. Does it happen once ("offline"), or per query?  
Deliverable: A short write-up (markdown cell in the notebook), ~2-4 sentences per 
component covering the three questions above. 
 

Answers:
1. 